## Advanced Rag System

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# 1. Navigate to project root
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
os.chdir(project_root)
sys.path.insert(0, str(project_root))

load_dotenv()

# 2. Load centralized config & embedding model
from src.services.llm_services import load_config, get_text_embeddings

config = load_config("src/config/config.yaml")
embeddings = get_text_embeddings(config)

print(f"📂 Project Root: {os.getcwd()}")
print(f"✅ Embedding Model: {config['text_emb_provider']} / {config['text_emb_model']}")

c:\Users\User\Desktop\Operation Legder-Mind\src\services\llm_services.py:133: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📂 Project Root: c:\Users\User\Desktop\Operation Legder-Mind
✅ Embedding Model: sbert / sentence-transformers/all-MiniLM-L6-v2


In [2]:
import weaviate
from langchain_weaviate import WeaviateVectorStore

# 1. Connect to local Weaviate running in Docker
client = weaviate.connect_to_local()
INDEX_NAME = "FinancialReportChunks"

if not client.is_ready():
    raise ConnectionError("❌ Local Weaviate is not running. Start Docker container first!")

# 2. Connect to the existing collection (No re-indexing required!)
vectorstore = WeaviateVectorStore(
    client=client,
    index_name=INDEX_NAME,
    text_key="text",
    embedding=embeddings
)

# 3. Check collection count
collection = client.collections.get(INDEX_NAME)
count = collection.aggregate.over_all(total_count=True).total_count

print(f"✅ Connected to existing Weaviate collection '{INDEX_NAME}'")
print(f"📊 Total indexed chunks available: {count}")

simsimd not available — falling back to SciPy/NumPy implementation for vector math. To enable the accelerated path, install the optional dependency 'simsimd' (note: simsimd may require newer glibc on some systems).


✅ Connected to existing Weaviate collection 'FinancialReportChunks'
📊 Total indexed chunks available: 543


In [3]:
query = "What is Uber's core mission statement as described in the 2024 annual report?"

# Perform similarity search
docs = vectorstore.similarity_search(query, k=3)

print("=" * 60)
print(f"🔍 QUERY: {query}")
print("=" * 60)

for i, doc in enumerate(docs, 1):
    page = doc.metadata.get("page", "N/A")
    splitter = doc.metadata.get("splitter", "N/A")
    
    print(f"\n🎯 [Rank #{i}] (SEC 10-K Page: {page} | Strategy: {splitter})")
    print("-" * 60)
    print(doc.page_content.strip())
    print("-" * 60)

🔍 QUERY: What is Uber's core mission statement as described in the 2024 annual report?

🎯 [Rank #1] (SEC 10-K Page: 1.0 | Strategy: recursive)
------------------------------------------------------------
Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The kind of people who are relentless about our  
mission to help people go anywhere and get anything and earn their way. 
Movement is what we power. It’s our lifeblood. It runs through our veins. It’s 
what gets us out of bed each morning. It pushes us to constantly reimagine 
how we can move better. For you. For all the places you want to go. For all the 
things you want to get. For all the ways you want to earn. Across the entire 
world. In real time. At the incredible speed of now.
------------------------------------------------------------

🎯 [Rank #2] (SEC 10-K Page: 11.0 | Strategy: recursive)
------------------------------------------------------------
Employees have access to an in

## Hybrid Search(Dense(vector search) +Sparce(BM25))

In [ ]:
from collections import defaultdict
from weaviate.classes.query import HybridFusion
from typing import List

if not client.is_connected():
    client.connect()

def rrf_fusion(dense_results, bm25_results, k=60, top_n=3) -> List:
    """
    Reciprocal Rank Fusion (RRF) - combines dense and sparse retrieval.
    
    RRF Formula: score(d) = Σ 1/(k + rank(d)) for each ranking
    
    - k=60 is the default constant (from original paper)
    - Higher k → more weight to lower-ranked documents
    - Lower k → more weight to top-ranked documents
    
    Args:
        dense_docs: Results from dense (vector) retrieval
        bm25_results: Results from BM25 (sparse) retrieval
        k: RRF constant (default 60, typical range 1-100)
        
    Returns:
        Fused results sorted by RRF score (higher = more relevant)
    """
    rrf_scores  = defaultdict(float)
    doc_map={}

    #1. Score dense vector ranks
    for rank, doc in enumerate(dense_results,1):
        doc_id= doc.metadata.get("page_content_hash", doc.page_content[:50])
        doc_map[doc_id] = doc
        rrf_scores[doc_id] += 1.0 / (k + rank)

    # 2. Score BM25 keyword ranks
    for rank, doc in enumerate(bm25_results, 1):
        doc_id = doc.metadata.get("page_content_hash", doc.page_content[:50])
        doc_map[doc_id] = doc
        rrf_scores[doc_id] += 1.0 / (k + rank)
    
    # 3. Sort by fused RRF score descending
    sorted_docs = sorted(rrf_scores.items(), key=lambda item:item[1], reverse=True)

    # 4. Return top_n fused documents with their score
    fused_results = []
    for doc_id, score in sorted_docs[:top_n]:
        fused_results.append((doc_map[doc_id], score))
    return fused_results



### 1. Execute Manual RRF Hybrid Search

In [ ]:
from langchain_core.documents import Document

query = "What was Uber's GAAP operating income in 2024?"

# 1. Dense Vector Retrieval (Top 10)
dense_docs = vectorstore.similarity_search(query, k=10)

# 2. Sparse BM25 Keyword Retrieval (Top 10)
bm25_raw = collection.query.bm25(query=query, limit=10)
bm25_docs = [
    Document(
        page_content=obj.properties.get("text", ""),
        metadata={
            "page": obj.properties.get("page"),
            "source": obj.properties.get("source"),
            "splitter": obj.properties.get("splitter")
        }
    )
    for obj in bm25_raw.objects
]

# 3. Fuse rankings using your rrf_fusion function (k=60)
fused_results = rrf_fusion(dense_docs, bm25_docs, k=60, top_n=3)

print("=" * 65)
print(f"🔍 MANUAL RRF HYBRID SEARCH: '{query}'")
print("=" * 65)

for rank, (doc, rrf_score) in enumerate(fused_results, 1):
    page = doc.metadata.get("page", "N/A")
    print(f"\n🏆 [Rank #{rank}] | RRF Score: {rrf_score:.5f} | Page: {page}")
    print("-" * 65)
    print(doc.page_content.strip()[:250] + "...")
    print("-" * 65)

### 2. Execute Native Weaviate Hybrid Search (Built-in RRF)

In [ ]:
# Embed query vector for Weaviate's native hybrid search
query_vector = embeddings.embed_query(query)

# Native Weaviate Hybrid Search using Reciprocal Rank Fusion (HybridFusion.RANKED)
native_hybrid_response = collection.query.hybrid(
    query=query,                       # Sparse / BM25 keyword search
    vector=query_vector,               # Dense / SBERT vector search
    alpha=0.5,                         # 0.5 = Balanced dense & sparse
    fusion_type=HybridFusion.RANKED,   # 👈 Weaviate's native RRF fusion
    limit=3,
    return_metadata=["score"]
)

print("=" * 65)
print(f"⚡ NATIVE WEAVIATE HYBRID SEARCH (RRF): '{query}'")
print("=" * 65)

for i, obj in enumerate(native_hybrid_response.objects, 1):
    score = obj.metadata.score if obj.metadata else 0.0
    page = obj.properties.get("page", "N/A")
    print(f"\n🏆 [Rank #{i}] | Native RRF Score: {score:.5f} | Page: {page}")
    print("-" * 65)
    print(obj.properties.get("text", "").strip()[:250] + "...")
    print("-" * 65)